# HW04-ICA :: Part A (PyTorch CNN on MNIST)

COSC 6373

Adam Nelson-Archer, 2140122


## Description / Tasks

The MNIST dataset contains 28×28 grayscale images of handwritten digits (0–9). The goal is to train a simple CNN classifier.

Tasks for Part A:
- Write the code needed to load the MNIST dataset into a PyTorch CNN model.
- Create a base CNN model (**Convolutional Layer → MaxPooling → Flatten → Dropout → Fully Connected layer**).


In [ ]:
from __future__ import annotations

import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


SEED = 6373
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reproducibility (best-effort)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("CWD:", os.getcwd())

In [ ]:
# Paths (relative to this notebook folder)
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR / "data").resolve()
ARTIFACTS_DIR = (NOTEBOOK_DIR / "artifacts").resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

In [ ]:
# MNIST data loaders
# Standard MNIST normalization values
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ]
)

train_ds = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))
x0, y0 = train_ds[0]
print("One sample tensor shape:", tuple(x0.shape), "label:", y0)

In [ ]:
# Visual sanity-check (first batch)
images, labels = next(iter(train_loader))

# Un-normalize for display
mean, std = 0.1307, 0.3081
images_disp = (images * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 6, figsize=(10, 3))
axes = axes.ravel()
for i in range(12):
    axes[i].imshow(images_disp[i].squeeze(0), cmap="gray")
    axes[i].set_title(str(int(labels[i])))
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class BaseMNISTCNN(nn.Module):
    """Base CNN for MNIST.

    Required pipeline (HW spec):
    Convolutional Layer -> MaxPooling -> Flatten -> Dropout -> Fully Connected layer
    """

    def __init__(self, num_classes: int = 10):
        super().__init__()

        # 1x28x28 -> 16x24x24 (kernel=5, no padding)
        self.conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5)

        # 16x24x24 -> 16x12x12
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Flatten: 16 * 12 * 12 = 2304
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(16 * 12 * 12, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = BaseMNISTCNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", f"{n_params:,}")

In [ ]:
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits.detach(), y))

    return float(np.mean(losses)), float(np.mean(accs))


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, y))

    return float(np.mean(losses)), float(np.mean(accs))

In [ ]:
# Train
EPOCHS = 5
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
}

best_test_acc = -1.0
best_path = ARTIFACTS_DIR / "mnist_cnn_best.pt"

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), best_path)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_loss:.4f}, acc={train_acc:.4f} | "
        f"test loss={test_loss:.4f}, acc={test_acc:.4f}"
    )

print("\nBest test acc:", f"{best_test_acc:.4f}")
print("Saved best weights to:", best_path)
print("Elapsed (s):", f"{(time.time() - t0):.1f}")

In [ ]:
# Plot learning curves
epochs = np.arange(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["test_loss"], label="test")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["test_acc"], label="test")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

plot_path = ARTIFACTS_DIR / "learning_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

In [ ]:
@torch.no_grad()
def confusion_matrix_torch(model: nn.Module, loader: DataLoader, device: torch.device, n_classes: int = 10) -> torch.Tensor:
    model.eval()
    cm = torch.zeros((n_classes, n_classes), dtype=torch.int64)
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        preds = model(x).argmax(dim=1)
        for t, p in zip(y.view(-1), preds.view(-1)):
            cm[int(t), int(p)] += 1
    return cm


# Load best weights and compute confusion matrix on test set
best_model = BaseMNISTCNN().to(device)
best_model.load_state_dict(torch.load(best_path, map_location=device))
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
cm = confusion_matrix_torch(best_model, test_loader, device)

print("Best model test loss:", f"{test_loss:.4f}")
print("Best model test acc:", f"{test_acc:.4f}")
print("Confusion matrix shape:", tuple(cm.shape))

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm.cpu().numpy(), cmap="Blues")
ax.set_title("Confusion Matrix (Test)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

cm_path = ARTIFACTS_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=150)
plt.show()

print("Saved:", cm_path)

## Model + Training Summary (ready to submit)

**Model (base CNN as required)**
- Convolutional Layer: `Conv2d(1 → 16, kernel_size=5)` + ReLU
- MaxPooling: `MaxPool2d(2×2)`
- Flatten: `16×12×12 = 2304`
- Dropout: `p=0.5`
- Fully Connected: `Linear(2304 → 10)` (logits)

**Training**
- Optimizer: Adam, learning rate (1e-3)
- Loss: CrossEntropyLoss
- Epochs: 5
- Batch size: 64

**Outputs saved** (in `HW4/Part_A/artifacts/`)
- `mnist_cnn_best.pt`
- `learning_curves.png`
- `confusion_matrix.png`


## Report Notes (fill after you run the notebook)

- Final best **test accuracy**: (copy the printed value)
- Anything notable in the confusion matrix? (which digits confuse each other?)
- Any parameter changes you tried (epochs, learning rate, dropout) and the effect.

## Export to PDF

In Jupyter/VSCode notebooks:
- Run all cells so outputs/figures appear.
- Export to PDF (or export to HTML then print-to-PDF).


## Acknowledgment

I used a coding assistant (ChatGPT, GPT‑5.2) to help scaffold and organize this notebook.
